# Capa Gold

## Se Transforman y agregan datos procesados desde la capa Silver (loans_enriched) para obtener insights útiles para el sector de Analisis en este caso.

## Se crean metricas y se guardan en capa gold para posteriormente ser analizadas o revisadas

### Las creaciones incluyen: 
- Tabla enriquecida general (joins entre tablas)
- Metricas de clientes
- Metricas de prestamos
- Promedios por tipo de cuenta bancaria
- Cantidad de pagos unicos por cliente
- Informe de calidad de datos
  

## 1 - Importacion de modulos
Se importan los modulos especificos

In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *


## - Verificamos la existencia de la tabla enriquecida en el metastore (capa_silver)

In [0]:
gold_enriched_df= spark.sql(f"""select * from capa_silver.tables.loans_enriched""")

In [0]:
display(gold_enriched_df)

Creamos tabla temporal enriquecida desde Silver

In [0]:
gold_enriched_df.createOrReplaceTempView("gold_enriched")

## 2 - Genero un _data mart _ o tabla resumen de métricas a nivel de cliente con:
- Total de préstamos que tiene o tuvo un cliente
- Monto total que el cliente ya pagó en todos sus préstamos
- Si el cliente tiene al menos un préstamo en mora
- Fecha de apertura de su primera cuenta bancaria.
- Lista de tipos de cuentas que tiene el cliente
- Edad del cliente
- Si el cliente tiene teléfono registrado
- Si tiene email registrado.

In [0]:
customers_metrics_df = spark.sql(
    """
    SELECT
        customer_id,
        name,
        city,
        COUNT(DISTINCT loan_id) AS total_loans,
        SUM(loan_amount) AS total_loan_amount,
        SUM(amount_paid) AS total_amount_paid,
        MAX(CASE WHEN status = 'en mora' THEN 1 ELSE 0 END) AS has_mora,
        MIN(opened_date) AS first_opened_date,
        collect_set(class_of_account) AS account_types,
        MAX(age) AS age,
        MAX(has_phone) AS has_phone,
        MAX(has_email) AS has_email
    FROM
        capa_silver.tables.loans_enriched
    GROUP BY
        customer_id,
        name,
        city
    """
)

display(customers_metrics_df)


## 3 - Genero un _data mart _ o tabla resumen de métricas a nivel de prestamo-cliente con:
- Suma total del dinero que se pagó para ese préstamo.
- Porcentaje del prestamo pagado (total_pagado / monto_del_prestamo)
- Calcula cuántos días pasaron desde que se otorgó el préstamo (start_date) hasta hoy. En este caso al ser fechas caducadas en el estudio no se toman en cuenta, pero es una metrica importante en sector bancario

In [0]:
loans_metrics_df = spark.sql(
    """
    SELECT
        loan_id,
        customer_id,
        start_date,
        loan_amount,
        status,
        SUM(amount_paid) AS total_paid,
        SUM(amount_paid) / FIRST(loan_amount) AS pct_paid,
        COUNT(CASE WHEN is_amount_paid_null = true THEN 1 END) AS missing_payments,
        DATEDIFF(CURRENT_DATE(), start_date) AS days_since_loan_start
    FROM
        capa_silver.tables.loans_enriched
    GROUP BY
    all
"""
)
display(loans_metrics_df)



## 4 - Genero un _data mart _ o tabla resumen de métricas a nivel de cuenta bancaria con:
- agrupacion por class_of_account
- Cuantos clientes únicos hay en cada clase de cuenta
- Calcula el saldo promedio de las cuentas de ese tipo
- Cuenta cuántos préstamos distintos hay en ese tipo de cuenta.
- Tasa de interés promedio de los préstamos asociados a cuentas de esa clase.

In [0]:
account_class_analytics_df = spark.sql(
"""
    SELECT
        class_of_account,
        COUNT(DISTINCT customer_id) AS customer_count,
        AVG(balance) AS avg_balance,
        COUNT(DISTINCT loan_id) AS total_loans,
        AVG(interest_rate) AS avg_interest_rate
    FROM
        capa_silver.tables.loans_enriched
    GROUP BY
        class_of_account
"""
)
display(account_class_analytics_df)


## 5 - Genero un _data mart _ o tabla resumen de métricas a nivel de pagos con:

- Agrupación por customer_id
- Cuenta la cantidad de pagos únicos que hizo cada cliente.
- Calcula el monto promedio pagado por pago para cada cliente.
- Indicador de porcentaje de monto de pagos 
_Nulo en avg_payment_amount indica que no existen pagos válidos para ese cliente, por lo tanto no se puede calcular el promedio del monto pagado_
- Cuánto se demora en promedio el pago desde que se registra la transacción.

_Días negativos en avg_days_delay indican que el pago se registró después de la fecha de la transacción, lo que puede señalar inconsistencias en las fechas o diferencias en el momento de registro. Como vimos habia 2 registros con fechas erroneas entonces tenemos que contactar con el banco para solucionarlos en silver_ 

(explicado en capa_silver notebook)


In [0]:

# agrupa por customer_id
# total_payments:= Cuenta la cantidad de pagos únicos que hizo cada cliente.
# avg_payment_amount : Calcula el monto promedio pagado por pago para cada cliente.
# Nulo en avg_payment_amount indica que no existen pagos válidos para ese cliente, por lo tanto no se puede calcular el promedio del monto pagado
# avg_days_delay : cuánto se demora en promedio el pago desde que se registra la transacción.

# Días negativos en avg_days_delay indican que el pago se registró después de la fecha de la transacción, lo que puede señalar inconsistencias en las fechas o diferencias en el momento de registro. Como vimos habia 2 registros con fechas erroneas entonces tenemos que contactar con el banco para solucionarlos en silver
payments_summary_df = spark.sql(
    """
    SELECT
        customer_id,
        COUNT(DISTINCT payment_id) AS total_payments,
        AVG(amount_paid) AS avg_payment_amount,
        AVG(DATEDIFF(transaction_date, payment_date)) AS avg_days_delay
    FROM
        capa_silver.tables.loans_enriched
    GROUP BY
        customer_id 
    """
)
display(payments_summary_df)



## 6 - Esta consulta genera un informe de calidad de datos que ayudan a entender la integridad de los datos

missing_start_dates : Cuenta cuántos registros no tienen fecha de inicio (start_date nulo).

missing_amount_paid : Cuenta registros con monto pagado faltante (amount_paid nulo).

customers_without_phone: Cuenta clientes que no tienen teléfono (has_phone = false).

accounts_without_opened_date : Cuenta cuentas que no tienen fecha de apertura (has_opened_date = false).

In [0]:
data_quality_report_df = spark.sql(
    """
    SELECT
        COUNT(CASE WHEN start_date IS NULL THEN 1 END) AS missing_start_dates,
        COUNT(CASE WHEN amount_paid IS NULL THEN 1 END) AS missing_amount_paid,
        COUNT(CASE WHEN has_phone = false THEN 1 END) AS customers_without_phone,
        COUNT(CASE WHEN has_opened_date = false THEN 1 END) AS accounts_without_opened_date
    FROM
        capa_silver.tables.loans_enriched
    """
)
display(data_quality_report_df)


## 7 - Creación de tablas temporales de cada DataFrame

In [0]:
customers_metrics_df.createOrReplaceTempView("customer_metrics")
loans_metrics_df.createOrReplaceTempView("loans_metrics")
account_class_analytics_df.createOrReplaceTempView("account_class_analytics")
payments_summary_df.createOrReplaceTempView("payments_summary")
data_quality_report_df.createOrReplaceTempView("data_quality_report")

## 9 - Creamos Schemas en el catalogo especifico (capa_gold.tables) y luego listamos cada tabla dentro del mismo catalogo mediante iteracion _FOR_

In [0]:
# Creacion de esquema y tabla en silver de tabla enriquecida

spark.sql(f"""
          CREATE SCHEMA IF NOT EXISTS capa_gold.tables
          """)

# Este código crea tablas Delta Lake dentro de Databricks, en el metastore predeterminado, mediante un for iterando los nombres de las tablas
# EXPLICAR DONDE SE GUARDAN Y COMO (DELTA TABLES)


table_names = ["gold_enriched", "customer_metrics", "loans_metrics", "account_class_analytics", "payments_summary", "data_quality_report"]

for table in table_names:
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS capa_gold.tables.{table}
        AS SELECT * FROM {table}
    """)    

## 10 - Guardamos en formato delta todos los DataSets ya limpios y transformados

Se uso un guardado simple no iterativo (_sin FOR_) para la visualizacion nativa del metodo de guardado

In [0]:
# guardo finalmente todas
gold_enriched_df.write.format("delta").mode("overwrite").save(f"abfss://gold@mistorageprincipal.dfs.core.windows.net/delta_tables/gold_enriched")
customers_metrics_df.write.format("delta").mode("overwrite").save(f"abfss://gold@mistorageprincipal.dfs.core.windows.net/delta_tables/customers_metrics")
loans_metrics_df.write.format("delta").mode("overwrite").save(f"abfss://gold@mistorageprincipal.dfs.core.windows.net/delta_tables/loans_metrics")
account_class_analytics_df.write.format("delta").mode("overwrite").save(f"abfss://gold@mistorageprincipal.dfs.core.windows.net/delta_tables/account_class_analytics")
payments_summary_df.write.format("delta").mode("overwrite").save(f"abfss://gold@mistorageprincipal.dfs.core.windows.net/delta_tables/payments_summary")
data_quality_report_df.write.format("delta").mode("overwrite").save(f"abfss://gold@mistorageprincipal.dfs.core.windows.net/delta_tables/data_quality_report")


### FINALIZACION CAPA _GOLD_
---
